# BetaTesting Gaussian 256 Random-5 Batch

This notebook searches a 256-combination grid on 5 reproducibly random raw images, chooses the best average parameter set, then processes every raw image with that fixed set. The iterated variables follow the thesis setup: homomorphic `gH`, residue reconstruction `beta`, gamma correction, and CLAHE clip limit.

In [10]:
from pathlib import Path
import csv
import gc
import itertools
import random
import sys
import time
from collections import Counter, defaultdict

try:
    import cv2
    import numpy as np
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Missing notebook dependency. Run this notebook with the project .venv '
        'interpreter: .venv/Scripts/python.exe'
    ) from exc

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'image_pipeline.py').exists():
    REPO_ROOT = Path('C:/Users/wonga/repo/pace_implementation')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from image_pipeline import ImageProcessingPipeline, PipelineConfig

print(f'Python: {sys.executable}')
print(f'Repo: {REPO_ROOT}')
print(f'OpenCV: {cv2.__version__}')

Python: c:\Users\wonga\repo\pace_implementation\.venv\Scripts\python.exe
Repo: c:\Users\wonga\repo\pace_implementation
OpenCV: 4.13.0


## Configuration

In [11]:
INPUT_DIR = Path('C:/Users/wonga/Downloads/data skripsi/BetaTesting Undiksha Mentah/raw_data')
GAIN_PATH = REPO_ROOT / 'datacitra' / 'Gain' / 'Trx' / '90_40_0,50.mdn'
DARK_PATH = REPO_ROOT / 'datacitra' / 'Dark' / 'Trx' / 'dark.mdn'
CALIBRATION_PATH = REPO_ROOT / 'datacitra' / 'Kalibrasi' / 'trx_44_35.npz'

OUTPUT_DIR = REPO_ROOT / 'output' / 'betatesting_gaussian256_random5'
IMAGE_OUTPUT_DIR = OUTPUT_DIR / 'images'
SAMPLE_GRID_CSV = OUTPUT_DIR / 'sample_grid_scores.csv'
BEST_PARAMETERS_CSV = OUTPUT_DIR / 'best_parameters.csv'
BATCH_RESULTS_CSV = OUTPUT_DIR / 'batch_results.csv'
SAMPLE_LIST_TXT = OUTPUT_DIR / 'random5_files.txt'

RANDOM_SEED = 42
SAMPLE_SIZE = 5
EXPECTED_RAW_COUNT = 43
EXPECTED_IMAGE_SHAPE = (3000, 4096)

# Thesis iteration grid:
# gH maps to the Gaussian homomorphic high-frequency gain (`rh`).
# beta maps to the residue reconstruction weight (`denoise_beta`).
FIXED_D0 = 40
FIXED_RL = 0.99
GH_VALUES = [2.5]
BETA_VALUES = [1.0]
GAMMA_VALUES = [0.8]
CLIP_LIMIT_VALUES = [3.0]
TILE_GRID_SIZE_VALUES = [(8, 8)]

# FABEMD configuration used by PipelineConfig when decomposition_method='fabemd'.
# Keep max_sift low for full-detector batch runs; raise it for slower, more refined BIMFs.
FABEMD_MAX_SIFT = 1
FABEMD_SD_THRESHOLD = 0.2
FABEMD_MIN_EXTREMA = 5
FABEMD_MAX_BIMFS = 20
FABEMD_WINDOW_SIZE_CAP = 2000
FABEMD_EXTREMA_WINDOW = 3
FABEMD_INITIAL_WINDOW_SIZE = None
FABEMD_WINDOW_GROWTH_RATE = 2.0

PARAMETER_COMBINATIONS = list(itertools.product(
    GH_VALUES,
    BETA_VALUES,
    GAMMA_VALUES,
    CLIP_LIMIT_VALUES,
    TILE_GRID_SIZE_VALUES,
))


BASE_CONFIG = PipelineConfig(
    gain_img_path=str(GAIN_PATH),
    dark_img_path=str(DARK_PATH),
    calibration_path=str(CALIBRATION_PATH),
    output_dir=str(OUTPUT_DIR),
    processing_mode='full',
    homomorphic_method='gaussian',
    decomposition_method='fabemd',
    fabemd_max_sift_iterations=FABEMD_MAX_SIFT,
    fabemd_sd_threshold=FABEMD_SD_THRESHOLD,
    fabemd_min_extrema=FABEMD_MIN_EXTREMA,
    fabemd_max_bimfs=FABEMD_MAX_BIMFS,
    fabemd_window_size_cap=FABEMD_WINDOW_SIZE_CAP,
    fabemd_extrema_window=FABEMD_EXTREMA_WINDOW,
    fabemd_initial_window_size=FABEMD_INITIAL_WINDOW_SIZE,
    fabemd_window_growth_rate=FABEMD_WINDOW_GROWTH_RATE,
    d0_values=[FIXED_D0],
    rh_values=GH_VALUES,
    rl_values=[FIXED_RL],
    gamma_values=GAMMA_VALUES,
    clip_limit_values=CLIP_LIMIT_VALUES,
    tile_grid_size_values=TILE_GRID_SIZE_VALUES,
    denoise_beta=BETA_VALUES[0],
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Total parameter combinations: {len(PARAMETER_COMBINATIONS)}')
print(
    'FABEMD config: '
    f'max_sift={FABEMD_MAX_SIFT}, sd={FABEMD_SD_THRESHOLD}, '
    f'min_extrema={FABEMD_MIN_EXTREMA}, max_bimfs={FABEMD_MAX_BIMFS}, '
    f'window_cap={FABEMD_WINDOW_SIZE_CAP}, extrema_window={FABEMD_EXTREMA_WINDOW}, '
    f'initial_window={FABEMD_INITIAL_WINDOW_SIZE}, growth={FABEMD_WINDOW_GROWTH_RATE}'
)
print(f'Output directory: {OUTPUT_DIR}')

Total parameter combinations: 1
FABEMD config: max_sift=1, sd=0.2, min_extrema=5, max_bimfs=20, window_cap=2000, extrema_window=3, initial_window=None, growth=2.0
Output directory: c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5


## Helpers

In [12]:
SEARCH_FIELDNAMES = [
    'sample_index', 'file', 'param_index', 'fixed_d0', 'gH', 'fixed_rL',
    'beta', 'gamma', 'clip_limit', 'tile_grid_rows', 'tile_grid_cols', 'cii', 'entropy',
    'eme', 'total_score', 'elapsed_seconds',
]

BEST_FIELDNAMES = [
    'param_index', 'fixed_d0', 'gH', 'fixed_rL', 'beta', 'gamma', 'clip_limit',
    'tile_grid_rows', 'tile_grid_cols', 'sample_count', 'average_cii',
    'average_entropy', 'average_eme', 'average_total_score',
]

BATCH_FIELDNAMES = [
    'batch_index', 'file', 'output_path', 'status', 'error', 'param_index',
    'fixed_d0', 'gH', 'fixed_rL', 'beta', 'gamma', 'clip_limit', 'tile_grid_rows',
    'tile_grid_cols', 'cii', 'entropy', 'eme', 'total_score',
    'elapsed_seconds',
]


def make_pipeline():
    return ImageProcessingPipeline(BASE_CONFIG)


def elapsed_text(seconds):
    minutes, secs = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours >= 1:
        return f'{int(hours)}h {int(minutes)}m {secs:.1f}s'
    if minutes >= 1:
        return f'{int(minutes)}m {secs:.1f}s'
    return f'{secs:.1f}s'


def read_image_summary(path):
    image = cv2.imread(str(path), -1)
    if image is None:
        raise FileNotFoundError(f'Could not load image: {path}')
    summary = {
        'path': str(path),
        'shape': tuple(image.shape),
        'dtype': str(image.dtype),
        'min': int(image.min()),
        'max': int(image.max()),
    }
    del image
    return summary


def write_csv(path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def pipeline_params(params):
    gH, beta, gamma, clip_limit, tile_grid_size = params
    return (FIXED_D0, gH, FIXED_RL, gamma, clip_limit, tile_grid_size), float(beta)


def set_residue_beta(pipeline, beta):
    pipeline.config.denoise_beta = float(beta)
    pipeline.nonlinear_filter.beta = float(beta)


def process_single_notebook_params(pipeline, params, reference_image, bimfs, energies, residue):
    pipeline_param_tuple, beta = pipeline_params(params)
    set_residue_beta(pipeline, beta)
    return pipeline._process_single_params(
        pipeline_param_tuple,
        reference_image,
        bimfs,
        energies,
        residue,
    )


def parameter_columns(param_index, params):
    gH, beta, gamma, clip_limit, tile_grid_size = params
    return {
        'param_index': int(param_index),
        'fixed_d0': FIXED_D0,
        'gH': gH,
        'fixed_rL': FIXED_RL,
        'beta': beta,
        'gamma': gamma,
        'clip_limit': clip_limit,
        'tile_grid_rows': int(tile_grid_size[0]),
        'tile_grid_cols': int(tile_grid_size[1]),
    }


def search_score_row(sample_index, proj_path, param_index, params, result, elapsed_seconds):
    row = {
        'sample_index': int(sample_index),
        'file': proj_path.name,
        'cii': float(result.cii),
        'entropy': float(result.entropy),
        'eme': float(result.eme),
        'total_score': float(result.total_score),
        'elapsed_seconds': round(float(elapsed_seconds), 3),
    }
    row.update(parameter_columns(param_index, params))
    return row


def preprocess_for_pipeline(pipeline, proj_path):
    bundle = {
        'proj_img': None,
        'gain_img': None,
        'dark_img': None,
        'ffc_img': None,
        'calibrated_img': None,
        'bimfs': None,
        'energies': None,
        'residue': None,
    }
    bundle['proj_img'], bundle['gain_img'], bundle['dark_img'] = pipeline.load_images(
        str(proj_path), str(GAIN_PATH), str(DARK_PATH)
    )
    bundle['ffc_img'] = pipeline.apply_ffc(
        bundle['proj_img'], bundle['gain_img'], bundle['dark_img']
    )
    bundle['calibrated_img'] = pipeline.apply_spatial_calibration(
        bundle['ffc_img'], str(CALIBRATION_PATH)
    )
    bundle['bimfs'], bundle['energies'], bundle['residue'] = pipeline.decompose_image(
        bundle['calibrated_img']
    )
    return bundle


def cleanup_bundle(pipeline, bundle):
    if not bundle:
        return
    pipeline._cleanup(
        bundle.get('proj_img'),
        bundle.get('gain_img'),
        bundle.get('dark_img'),
        bundle.get('ffc_img'),
        bundle.get('calibrated_img'),
        bundle.get('bimfs'),
        bundle.get('energies'),
        bundle.get('residue'),
    )


def score_parameter_grid_for_file(proj_path, sample_index):
    pipeline = make_pipeline()
    bundle = None
    rows = []
    file_start = time.perf_counter()
    try:
        bundle = preprocess_for_pipeline(pipeline, proj_path)
        for param_index, params in enumerate(PARAMETER_COMBINATIONS, start=1):
            param_start = time.perf_counter()
            result = process_single_notebook_params(
                pipeline,
                params,
                bundle['calibrated_img'],
                bundle['bimfs'],
                bundle['energies'],
                bundle['residue'],
            )
            rows.append(search_score_row(
                sample_index, proj_path, param_index, params, result,
                time.perf_counter() - param_start,
            ))
            del result
            if param_index % 32 == 0 or param_index == len(PARAMETER_COMBINATIONS):
                print(
                    f'  {proj_path.name}: {param_index}/{len(PARAMETER_COMBINATIONS)} '
                    f'params in {elapsed_text(time.perf_counter() - file_start)}'
                )
                gc.collect()
        return rows
    finally:
        cleanup_bundle(pipeline, bundle)
        gc.collect()


def choose_best_parameter(search_rows):
    grouped = defaultdict(list)
    for row in search_rows:
        grouped[int(row['param_index'])].append(row)

    summary_rows = []
    for param_index in sorted(grouped):
        rows = grouped[param_index]
        params = PARAMETER_COMBINATIONS[param_index - 1]
        summary = parameter_columns(param_index, params)
        summary.update({
            'sample_count': len(rows),
            'average_cii': float(np.mean([float(row['cii']) for row in rows])),
            'average_entropy': float(np.mean([float(row['entropy']) for row in rows])),
            'average_eme': float(np.mean([float(row['eme']) for row in rows])),
            'average_total_score': float(np.mean([float(row['total_score']) for row in rows])),
        })
        summary_rows.append(summary)

    best_summary = max(
        summary_rows,
        key=lambda row: (float(row['average_total_score']), -int(row['param_index'])),
    )
    best_params = PARAMETER_COMBINATIONS[int(best_summary['param_index']) - 1]
    return best_summary, summary_rows, best_params


def process_file_with_fixed_params(proj_path, params, param_index, batch_index, output_path):
    pipeline = make_pipeline()
    bundle = None
    file_start = time.perf_counter()
    try:
        bundle = preprocess_for_pipeline(pipeline, proj_path)
        result = process_single_notebook_params(
            pipeline,
            params,
            bundle['calibrated_img'],
            bundle['bimfs'],
            bundle['energies'],
            bundle['residue'],
        )
        final_image = pipeline.normalize_and_resize(result.image)
        pipeline.save_image(final_image, str(output_path))
        row = {
            'batch_index': int(batch_index),
            'file': proj_path.name,
            'output_path': str(output_path),
            'status': 'ok',
            'error': '',
            'cii': float(result.cii),
            'entropy': float(result.entropy),
            'eme': float(result.eme),
            'total_score': float(result.total_score),
            'elapsed_seconds': round(float(time.perf_counter() - file_start), 3),
        }
        row.update(parameter_columns(param_index, params))
        del final_image, result
        return row
    finally:
        cleanup_bundle(pipeline, bundle)
        gc.collect()


def failed_batch_row(proj_path, params, param_index, batch_index, output_path, error, elapsed_seconds):
    row = {
        'batch_index': int(batch_index),
        'file': proj_path.name,
        'output_path': str(output_path),
        'status': 'failed',
        'error': repr(error),
        'cii': '',
        'entropy': '',
        'eme': '',
        'total_score': '',
        'elapsed_seconds': round(float(elapsed_seconds), 3),
    }
    row.update(parameter_columns(param_index, params))
    return row

## Validate Inputs And Select Random 5

In [13]:
required_paths = [INPUT_DIR, GAIN_PATH, DARK_PATH, CALIBRATION_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('Missing required paths: ' + ', '.join(str(path) for path in missing_paths))

raw_files = sorted(INPUT_DIR.glob('*.mdn'))
if len(raw_files) != EXPECTED_RAW_COUNT:
    raise AssertionError(f'Expected {EXPECTED_RAW_COUNT} raw files, found {len(raw_files)}')

raw_summaries = [read_image_summary(path) for path in raw_files]
raw_shape_counts = Counter((summary['shape'], summary['dtype']) for summary in raw_summaries)
if raw_shape_counts != Counter({(EXPECTED_IMAGE_SHAPE, 'uint16'): EXPECTED_RAW_COUNT}):
    raise AssertionError(f'Unexpected raw image shapes/dtypes: {raw_shape_counts}')

gain_summary = read_image_summary(GAIN_PATH)
dark_summary = read_image_summary(DARK_PATH)
for label, summary in [('gain', gain_summary), ('dark', dark_summary)]:
    if summary['shape'] != EXPECTED_IMAGE_SHAPE or summary['dtype'] != 'uint16':
        raise AssertionError(f'Unexpected {label} image summary: {summary}')

rng = random.Random(RANDOM_SEED)
sample_files = rng.sample(raw_files, SAMPLE_SIZE)
SAMPLE_LIST_TXT.write_text('\n'.join(str(path) for path in sample_files) + '\n', encoding='utf-8')

print(f'Raw files validated: {len(raw_files)}')
print(f'Gain summary: {gain_summary}')
print(f'Dark summary: {dark_summary}')
print(f'Parameter grid count: {len(PARAMETER_COMBINATIONS)}')
print('Random sample files:')
for path in sample_files:
    print(f'  - {path.name}')
print(f'Sample list written to: {SAMPLE_LIST_TXT}')

Raw files validated: 43
Gain summary: {'path': 'c:\\Users\\wonga\\repo\\pace_implementation\\datacitra\\Gain\\Trx\\90_40_0,50.mdn', 'shape': (3000, 4096), 'dtype': 'uint16', 'min': 0, 'max': 1222}
Dark summary: {'path': 'c:\\Users\\wonga\\repo\\pace_implementation\\datacitra\\Dark\\Trx\\dark.mdn', 'shape': (3000, 4096), 'dtype': 'uint16', 'min': 0, 'max': 72}
Parameter grid count: 1
Random sample files:
  - 8-KAA-08B_Thorax_PA 8-KAA-08B 90kV40mA0,50s -8_5_2024-9.28 PM [Administrator].mdn
  - 14-CCA-14B_Thorax_AP 14-CCA-14B 90kV40mA0,50s -8_5_2024-8.04 PM [Administrator].mdn
  - 1-IMA-01B_Thorax_AP 1-IMA-01B 90kV40mA0,50s -8_6_2024-2.29 AM [Administrator].mdn
  - 23-MBD-23A_Thorax_PA 23-MBD-23A 90kV40mA0,50s -8_5_2024-9.59 PM [Administrator].mdn
  - 21-KHY-32A_Thorax_PA 21-KHY-32A 90kV40mA0,50s -8_5_2024-11.43 PM [Administrator].mdn
Sample list written to: c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\random5_files.txt


## Smoke Test One File And One Parameter

In [14]:
smoke_path = sample_files[0]
smoke_param_index = 1
smoke_params = PARAMETER_COMBINATIONS[smoke_param_index - 1]
smoke_pipeline = make_pipeline()
smoke_bundle = None
smoke_result = None
smoke_start = time.perf_counter()

try:
    smoke_bundle = preprocess_for_pipeline(smoke_pipeline, smoke_path)
    smoke_result = process_single_notebook_params(
        smoke_pipeline,
        smoke_params,
        smoke_bundle['calibrated_img'],
        smoke_bundle['bimfs'],
        smoke_bundle['energies'],
        smoke_bundle['residue'],
    )
    print(f'Smoke file: {smoke_path.name}')
    print(f'Smoke params: {smoke_params}')
    print(f'CII={smoke_result.cii:.6f}, entropy={smoke_result.entropy:.6f}, EME={smoke_result.eme:.6f}')
    print(f'Total score={smoke_result.total_score:.6f}')
    print(f'Elapsed: {elapsed_text(time.perf_counter() - smoke_start)}')
finally:
    del smoke_result
    cleanup_bundle(smoke_pipeline, smoke_bundle)
    gc.collect()

2026-05-31 22:43:35,635 - INFO - Loading images...
2026-05-31 22:43:35,814 - INFO - Images loaded successfully.
2026-05-31 22:43:35,816 - INFO - Applying Flat Field Correction...
2026-05-31 22:44:11,224 - INFO - Flat Field Correction completed.
2026-05-31 22:44:11,231 - INFO - Applying Spatial Calibration...
2026-05-31 22:44:11,726 - INFO - Spatial Calibration completed.
2026-05-31 22:44:11,728 - INFO - Starting image decomposition (FABEMD)...
2026-05-31 22:44:11,731 - INFO - Starting FABEMD decomposition...


FABEMD: 7 BIMFs | window=511 | residual extrema: 80853

2026-05-31 22:45:26,906 - INFO - Stopping: residual has ≤ 5 extrema (4).
2026-05-31 22:45:26,911 - INFO - FABEMD decomposition completed — 8 BIMFs extracted.


FABEMD: 8 BIMFs | window=1023 | residual extrema: 4


2026-05-31 22:45:29,048 - INFO - Image decomposition completed.
2026-05-31 22:45:38,734 - INFO - Cleaning up memory...
2026-05-31 22:45:38,917 - INFO - Memory cleaned.


Smoke file: 8-KAA-08B_Thorax_PA 8-KAA-08B 90kV40mA0,50s -8_5_2024-9.28 PM [Administrator].mdn
Smoke params: (2.5, 1.0, 0.8, 3.0, (8, 8))
CII=1.000000, entropy=9.718405, EME=135.112354
Total score=145.830758
Elapsed: 2m 3.1s


## Run 256-Parameter Search On Random 5

In [15]:
all_search_rows = []
search_start = time.perf_counter()

for sample_index, sample_path in enumerate(sample_files, start=1):
    print(f'[{sample_index}/{len(sample_files)}] Searching {sample_path.name}')
    sample_rows = score_parameter_grid_for_file(sample_path, sample_index)
    all_search_rows.extend(sample_rows)
    write_csv(SAMPLE_GRID_CSV, all_search_rows, SEARCH_FIELDNAMES)
    best_sample_row = max(sample_rows, key=lambda row: float(row['total_score']))
    print(
        f'  best for sample: param #{best_sample_row["param_index"]}, '
        f'score={best_sample_row["total_score"]:.6f}'
    )
    print(f'  partial search CSV written to: {SAMPLE_GRID_CSV}')

print(f'Search rows: {len(all_search_rows)}')
print(f'Total search elapsed: {elapsed_text(time.perf_counter() - search_start)}')

2026-05-31 22:45:39,097 - INFO - Loading images...


[1/5] Searching 8-KAA-08B_Thorax_PA 8-KAA-08B 90kV40mA0,50s -8_5_2024-9.28 PM [Administrator].mdn


2026-05-31 22:45:39,339 - INFO - Images loaded successfully.
2026-05-31 22:45:39,341 - INFO - Applying Flat Field Correction...
2026-05-31 22:46:04,918 - INFO - Flat Field Correction completed.
2026-05-31 22:46:04,922 - INFO - Applying Spatial Calibration...
2026-05-31 22:46:05,085 - INFO - Spatial Calibration completed.
2026-05-31 22:46:05,086 - INFO - Starting image decomposition (FABEMD)...
2026-05-31 22:46:05,087 - INFO - Starting FABEMD decomposition...


FABEMD: 7 BIMFs | window=511 | residual extrema: 80853

2026-05-31 22:46:37,800 - INFO - Stopping: residual has ≤ 5 extrema (4).
2026-05-31 22:46:37,801 - INFO - FABEMD decomposition completed — 8 BIMFs extracted.


...

FABEMD: 7 BIMFs | window=511 | residual extrema: 12643

2026-05-31 22:50:47,451 - INFO - Stopping: residual has ≤ 5 extrema (3).
2026-05-31 22:50:47,452 - INFO - FABEMD decomposition completed — 8 BIMFs extracted.


FABEMD: 8 BIMFs | window=1023 | residual extrema: 3


2026-05-31 22:50:48,358 - INFO - Image decomposition completed.
2026-05-31 22:50:54,445 - INFO - Cleaning up memory...
2026-05-31 22:50:54,532 - INFO - Memory cleaned.


  21-KHY-32A_Thorax_PA 21-KHY-32A 90kV40mA0,50s -8_5_2024-11.43 PM [Administrator].mdn: 1/1 params in 59.1s
  best for sample: param #1, score=126.720561
  partial search CSV written to: c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\sample_grid_scores.csv
Search rows: 5
Total search elapsed: 5m 15.6s


## Choose Best Global Parameter

In [16]:
best_summary, parameter_summary_rows, best_params = choose_best_parameter(all_search_rows)
best_param_index = int(best_summary['param_index'])
# write_csv(BEST_PARAMETERS_CSV, [best_summary], BEST_FIELDNAMES)

print('Best global Gaussian parameters:')
for key in BEST_FIELDNAMES:
    print(f'  {key}: {best_summary[key]}')
print(f'Best parameter tuple: {best_params}')
print(f'Best parameter summary written to: {BEST_PARAMETERS_CSV}')

Best global Gaussian parameters:
  param_index: 1
  fixed_d0: 40
  gH: 2.5
  fixed_rL: 0.99
  beta: 1.0
  gamma: 0.8
  clip_limit: 3.0
  tile_grid_rows: 8
  tile_grid_cols: 8
  sample_count: 5
  average_cii: 1.0
  average_entropy: 10.1820650100708
  average_eme: 98.81550642406651
  average_total_score: 109.99757143413731
Best parameter tuple: (2.5, 1.0, 0.8, 3.0, (8, 8))
Best parameter summary written to: c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\best_parameters.csv


## Batch Process All Raw Images With Fixed Best Parameter

In [17]:
batch_rows = []
batch_start = time.perf_counter()

for batch_index, proj_path in enumerate(raw_files, start=1):
    output_path = IMAGE_OUTPUT_DIR / f'{proj_path.stem}_processed.tiff'
    file_start = time.perf_counter()
    print(f'[{batch_index}/{len(raw_files)}] Processing {proj_path.name}')
    try:
        row = process_file_with_fixed_params(
            proj_path,
            best_params,
            best_param_index,
            batch_index,
            output_path,
        )
        print(f'  saved: {output_path.name} score={row["total_score"]:.6f}')
    except Exception as exc:
        row = failed_batch_row(
            proj_path,
            best_params,
            best_param_index,
            batch_index,
            output_path,
            exc,
            time.perf_counter() - file_start,
        )
        print(f'  failed: {exc!r}')

    batch_rows.append(row)
    write_csv(BATCH_RESULTS_CSV, batch_rows, BATCH_FIELDNAMES)
    print(f'  batch CSV updated: {BATCH_RESULTS_CSV}')

print(f'Batch rows: {len(batch_rows)}')
print(f'Total batch elapsed: {elapsed_text(time.perf_counter() - batch_start)}')

2026-05-31 22:50:54,731 - INFO - Loading images...


[1/43] Processing 02-WCI-02B_Thorax_PA 02-WCI-02B 90kV40mA0,50s -8_6_2024-2.04 AM [Administrator].mdn


2026-05-31 22:50:54,916 - INFO - Images loaded successfully.
2026-05-31 22:50:54,917 - INFO - Applying Flat Field Correction...
2026-05-31 22:51:15,608 - INFO - Flat Field Correction completed.
2026-05-31 22:51:15,612 - INFO - Applying Spatial Calibration...
2026-05-31 22:51:15,761 - INFO - Spatial Calibration completed.
2026-05-31 22:51:15,763 - INFO - Starting image decomposition (FABEMD)...
2026-05-31 22:51:15,763 - INFO - Starting FABEMD decomposition...


FABEMD: 7 BIMFs | window=511 | residual extrema: 94664

2026-05-31 22:51:47,518 - INFO - Stopping: residual has ≤ 5 extrema (4).
2026-05-31 22:51:47,521 - INFO - FABEMD decomposition completed — 8 BIMFs extracted.


...

FABEMD: 7 BIMFs | window=511 | residual extrema: 1131

2026-06-01 00:01:51,081 - INFO - Stopping: residual has ≤ 5 extrema (4).
2026-06-01 00:01:51,082 - INFO - FABEMD decomposition completed — 8 BIMFs extracted.


FABEMD: 8 BIMFs | window=1023 | residual extrema: 4


2026-06-01 00:01:52,005 - INFO - Image decomposition completed.
2026-06-01 00:02:01,922 - INFO - Image saved to: c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\images\9-KSA-09B_Thorax_AP_2 9-KSA-09B 90kV40mA0,50s -8_5_2024-8.00 PM [Administrator]_processed.tiff
2026-06-01 00:02:01,932 - INFO - Cleaning up memory...
2026-06-01 00:02:02,017 - INFO - Memory cleaned.


  saved: 9-KSA-09B_Thorax_AP_2 9-KSA-09B 90kV40mA0,50s -8_5_2024-8.00 PM [Administrator]_processed.tiff score=104.667961
  batch CSV updated: c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\batch_results.csv
Batch rows: 43
Total batch elapsed: 1h 11m 7.5s


## Final Audit

In [18]:
written_images = sorted(IMAGE_OUTPUT_DIR.glob('*_processed.tiff'))
ok_rows = [row for row in batch_rows if row.get('status') == 'ok']
failed_rows = [row for row in batch_rows if row.get('status') != 'ok']

print(f'Processed image files written: {len(written_images)}')
print(f'Batch ok rows: {len(ok_rows)}')
print(f'Batch failed rows: {len(failed_rows)}')
print(f'Search CSV exists: {SAMPLE_GRID_CSV.exists()} -> {SAMPLE_GRID_CSV}')
print(f'Best parameter CSV exists: {BEST_PARAMETERS_CSV.exists()} -> {BEST_PARAMETERS_CSV}')
print(f'Batch CSV exists: {BATCH_RESULTS_CSV.exists()} -> {BATCH_RESULTS_CSV}')

if len(written_images) != len(raw_files):
    raise AssertionError(f'Expected {len(raw_files)} processed images, found {len(written_images)}')
if failed_rows:
    raise AssertionError(f'Batch failures found: {failed_rows[:3]}')

Processed image files written: 43
Batch ok rows: 43
Batch failed rows: 0
Search CSV exists: True -> c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\sample_grid_scores.csv
Best parameter CSV exists: False -> c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\best_parameters.csv
Batch CSV exists: True -> c:\Users\wonga\repo\pace_implementation\output\betatesting_gaussian256_random5\batch_results.csv
